# `ptof_obs_setup_seed`

## What this notebook does
Hand-maintained reference-data seeding and one-off maintenance for the observability pipeline.
It is **not part of the scheduled `obs_fresh_scan` job** — it's run manually, by a human, when a
new capability needs registering, a threshold needs recording, or a one-time cleanup/backfill is
needed. Every other notebook in this repo *reads* the tables this notebook seeds; nothing else
writes to them.

## Prod migration (2026-09-10)
This notebook seeds reference data for the 4 prod capabilities tracked via
`mq_gmdf_dp_prd.oil.ptof_primary__ai_shift_outputs`:
- `saa-display`, `situational-awareness`, `sev2-insights`, `summary`

Old dev capabilities (`saa_insight`, `sev2_insight`, `watchout_narratives`) are deactivated (not
deleted) for historical reference. Detectors dropped in the migration (latency, error rate,
hallucination, transport violations, prompt size, credential fastfail, rapid human correction)
have their output tables dropped in the cleanup cell and their threshold_basis entries marked
`not_applicable_prod`.

## Position in the pipeline
- **Not in any job DAG.** Run interactively/manually, occasionally, not on a schedule.
- **Downstream readers:** `ptof_obs_liveness_detection`, `ptof_obs_mal_output`,
  `ptof_obs_behavioral_correlation` all join against `capability_registry`.
  `ptof_obs_alert.ipynb` reads `threshold_basis` for triage context on Teams cards and
  creates/updates `obs_incidents`.

## Write safety
`capability_registry` and `threshold_basis` are seeded via `CREATE TABLE IF NOT EXISTS` +
a `LEFT ANTI JOIN` insert keyed on each table's natural key — re-running is always a no-op
against rows that already exist.

| Cell | Writes | Idempotent? |
|---|---|---|
| 1 | `capability_registry` | Yes — anti-join insert + deactivate old + update summary |
| 2 | `obs_incidents` schema | Yes — `CREATE TABLE IF NOT EXISTS` |
| 3 | `threshold_basis` | Yes — anti-join insert on check_name |
| 4 | orphaned tables (drop) | Yes — all `DROP TABLE/VIEW IF EXISTS` |

In [ ]:
%sql
-- SAFE to re-run: CREATE TABLE IF NOT EXISTS + anti-join insert, guarded per capability.
-- Seeds capability_registry with all historical rows (dev, now inactive) and the 4 active
-- prod capabilities. The deactivate UPDATE and summary UPDATE at the bottom are idempotent.
--
-- Prod capabilities (active = true):
--   saa-display          — SAA shift display output (~2,633 rows), 2h silence grace
--   situational-awareness — SAA situational awareness (~2,633 rows), 2h silence grace
--   sev2-insights        — SAA sev2 insights (~572 rows), irregular cadence → no silence grace
--   summary              — ISH EOS handover summary (~53 rows), 36h silence grace
--
-- is_groundable = false for all prod capabilities: hallucination detection is dropped because
-- the regex+similarity approach flags computed numbers as hallucinated (4.9% false positive rate).
CREATE TABLE IF NOT EXISTS mq_gmdf_dev.oil_obs.capability_registry (
    capability           STRING,
    is_generative        BOOLEAN,
    is_gxp_relevant      BOOLEAN,
    expected_min_daily   INT,
    required_fields      ARRAY<STRING>,
    owner                STRING,
    active               BOOLEAN,
    notes                STRING,
    silence_grace_hours  INT,
    is_groundable        BOOLEAN
);

-- Insert all capabilities (both historical dev and new prod) via anti-join so re-running is safe.
INSERT INTO mq_gmdf_dev.oil_obs.capability_registry
  (capability, is_generative, is_gxp_relevant, expected_min_daily, owner, active, notes,
   silence_grace_hours, is_groundable)
SELECT t.* FROM VALUES
  -- historical dev capabilities (inactive, kept for reference)
  ('dsa_batch_summary',  true,  true,  0,  'dsa-team',  false,
     'vertex26 · dev-only, deactivated for prod migration 2026-09-10', NULL, false),
  ('dsa_compare',        true,  false, 0,  'dsa-team',  false,
     'vertex26 · dev-only, deactivated for prod migration 2026-09-10', NULL, false),
  ('dsa_copilot',        true,  false, 20, 'dsa-team',  false,
     'vertex26 · dev-only, deactivated for prod migration 2026-09-10', NULL, false),
  ('dsa_copilot_step',   true,  false, 0,  'unassigned', false,
     'cortex · dev-only, deactivated for prod migration 2026-09-10', NULL, false),
  ('dsa_optimize',       true,  false, 10, 'dsa-team',  false,
     'dev-only, deactivated for prod migration 2026-09-10', NULL, false),
  ('dsa_session_summary',true,  false, 0,  'unassigned', false,
     'cortex · dev-only, deactivated for prod migration 2026-09-10', NULL, false),
  ('probe',              true,  false, 0,  'unassigned', false,
     'cortex · dev-only, deactivated for prod migration 2026-09-10', NULL, false),
  ('saa_insight',        true,  true,  5,  'saa-team',  false,
     'dev output_type. deactivated for prod migration 2026-09-10 — replaced by saa-display', 144, false),
  ('sev2_insight',       true,  true,  1,  'saa-team',  false,
     'dev output_type. deactivated for prod migration 2026-09-10 — replaced by sev2-insights', 144, false),
  ('watchout_narratives',true,  true,  0,  'ish-team',  false,
     'dev output_type. deactivated for prod migration 2026-09-10 — single call 2026-08-12', NULL, false),
  -- prod capabilities (active)
  ('saa-display',        true,  true,  100, 'saa-team',  true,
     'prod output_type=saa-display. ~2,633 rows. is_groundable=false (hallucination detector dropped).', 2, false),
  ('situational-awareness', true, true, 100, 'saa-team', true,
     'prod output_type=situational-awareness. ~2,633 rows. is_groundable=false.', 2, false),
  ('sev2-insights',      true,  true,  10,  'saa-team',  true,
     'prod output_type=sev2-insights. ~572 rows. silence_grace_hours=NULL (irregular cadence, '
     'natural 100+ hour gaps).', NULL, false),
  ('summary',            true,  true,  1,   'ish-team',  true,
     'prod output_type=summary. saa->ish-eos scheduler. is_groundable=false (no prompts in prod).', 36, false)
AS t(capability, is_generative, is_gxp_relevant, expected_min_daily, owner, active, notes,
     silence_grace_hours, is_groundable)
LEFT ANTI JOIN mq_gmdf_dev.oil_obs.capability_registry existing
  ON existing.capability = t.capability;

-- Deactivate old dev capabilities that may already exist with active=true from prior seeding.
UPDATE mq_gmdf_dev.oil_obs.capability_registry
SET active = false,
    is_groundable = false,
    notes = concat(coalesce(notes, ''), ' | deactivated for prod migration 2026-09-10')
WHERE capability IN ('saa_insight', 'sev2_insight', 'watchout_narratives')
  AND active = true;

-- Update existing summary row to prod settings (is_groundable=false, prod notes).
UPDATE mq_gmdf_dev.oil_obs.capability_registry
SET is_groundable = false,
    notes = 'prod output_type=summary. saa->ish-eos scheduler. is_groundable=false (no prompts in prod).'
WHERE capability = 'summary'
  AND is_groundable = true;

In [ ]:
%sql
-- SAFE to re-run: CREATE TABLE IF NOT EXISTS. This is the single table every detector's finding
-- ultimately lands in, and the only table ptof_obs_alert.ipynb's Teams-notify query reads from.
--
-- obs_incidents — append-only finding record.
-- MERGE keyed on (detector, source_row_id) so re-detecting the same finding updates rather than
-- duplicates. That gives dedup for free and makes acknowledgement stick across runs.
CREATE TABLE IF NOT EXISTS mq_gmdf_dev.oil_obs.obs_incidents (
    detector         STRING,        -- which detector produced this
    source_row_id    STRING,        -- the id being flagged; stable across runs
    capability       STRING,
    severity         STRING,        -- CRITICAL | WARN | INFO
    first_detected   TIMESTAMP,     -- set once, never updated
    last_detected    TIMESTAMP,     -- refreshed each time the finding is still present
    detection_count  BIGINT,        -- how many runs have seen it
    signal_payload   STRING,        -- JSON detail for triage
    notified_at      TIMESTAMP,     -- stamped when the alert reports it
    acknowledged_by  STRING,        -- set by a human, by hand
    acknowledged_at  TIMESTAMP,
    resolved_at      TIMESTAMP      -- set by hand once remediated
) CLUSTER BY (first_detected, detector);

In [ ]:
%sql
-- SAFE to re-run: CREATE TABLE IF NOT EXISTS + anti-join insert, guarded per check_name.
-- Seeds threshold_basis with documentation for every detection threshold in this system.
-- Retired detector thresholds are kept for historical reference with status = 'not_applicable_prod'.
-- New ETL pipeline thresholds added for the prod migration.
CREATE TABLE IF NOT EXISTS mq_gmdf_dev.oil_obs.threshold_basis (
    check_name STRING,
    threshold  STRING,
    basis      STRING,
    set_on     STRING,
    status     STRING
);

INSERT INTO mq_gmdf_dev.oil_obs.threshold_basis
SELECT t.* FROM VALUES
  -- active prod thresholds
  ('blank_output', '>3 blank_count AND >=10 total_calls AND >2% blank_rate, 6h window',
   'Three-part floor on blank_output_findings so an isolated blank response does not page. '
   'Feeds a CRITICAL Teams alert. Thresholds anchored to plausible-noise levels.',
   '2026-09-09', 'unvalidated'),
  ('etl_pipeline_staleness', 'max(run_timestamp) < now() - INTERVAL 30 MINUTES',
   'The upstream ETL refreshes ~19 tables every 10-15 min. If no run completes for 30 min, '
   'the SAA agent is running on stale data. Feeds a CRITICAL check in the alert notebook.',
   '2026-09-10', 'provisional'),
  ('etl_pipeline_failure', 'any failure in etl_pipeline_health (24h window)',
   'Any ETL task failure means at least one source table did not refresh. The agent''s outputs '
   'look normal but the underlying data is stale — capability_silence won''t fire.',
   '2026-09-10', 'provisional'),
  ('handover_delivery_new_failure',
   'any handover_delivery_failures row not yet recorded+acknowledged in obs_incidents',
   'Catches a single new occurrence — each one is a real shift handover that did not reach '
   'PFS3_ISH_SME@lists.lilly.com.',
   '2026-09-09', 'provisional'),
  ('handover_delivery_rate', 'failure_pct_7d > 20 AND attempts >= 10',
   'all-time baseline 9.3% (15 of 162 since Jun 19); currently 12.5% over 7d',
   '2026-08-20', 'provisional'),
  ('long_running_incident', 'first_detected <= now() - INTERVAL 3 DAYS',
   'Three days unacknowledged is a real signal. Age-based, not detection_count-based.',
   '2026-09-08', 'revised'),
  ('nightly_baseline_staleness', 'computed_at < now() - INTERVAL 36 HOURS',
   'Nightly baseline job runs once/day; 36h gives ~12h grace past the nightly schedule.',
   '2026-09-09', 'provisional'),
  ('pipeline_heartbeat', '0 rows in v_llm_bronze, trailing 2 hours',
   'Catches a total upstream ingestion outage that every other detector implicitly assumes '
   'cannot happen.',
   '2026-09-09', 'provisional'),
  ('response_baseline_min_rows', '>= 20 eligible rows per capability',
   'Guard on response_field_baseline: a capability with fewer than 20 eligible rows is excluded '
   'from the baseline rather than inferring a schema shape from too few samples.',
   '2026-09-05', 'provisional'),
  ('schema_field_missing', 'current_present = 0 AND baseline_presence_rate >= 0.2',
   'A field that appeared in >= 20% of baseline rows disappearing entirely is drift. Rarely- '
   'populated optional fields are excluded from noise.',
   '2026-09-09', 'provisional'),
  ('unacknowledged_critical', 'severity = CRITICAL AND acknowledged_at IS NULL AND resolved_at IS NULL',
   'Backstop rollup across every detector: catches anything already flagged that nobody has '
   'acted on.',
   '2026-09-09', 'provisional'),
  -- retired detector thresholds (kept for historical reference)
  ('capability_error_rate', 'error_rate = 1.0 & n>=5, or > 0.20 & n>=10',
   'observed 0.00 or 0.93-1.00; no middle ground in data', '2026-08-20', 'not_applicable_prod'),
  ('capability_error_rate_sustained', '0.50',
   'Bimodal failure distribution (0% or 93%+). Dropped: no success/error_msg in prod.',
   '2026-09-03', 'not_applicable_prod'),
  ('capability_silence dsa_copilot', 'silence_grace_hours = 26',
   'Dev-only capability, deactivated.', '2026-08-20', 'not_applicable_prod'),
  ('capability_silence dsa_optimize', 'silence_grace_hours = 2',
   'Dev-only capability, deactivated.', '2026-08-20', 'not_applicable_prod'),
  ('credential_outage', 'sum(fastfail_calls) > 0, current day',
   'Dev-specific model (demo-claude-sonnet-4-6-pwc-omi) does not exist in prod.',
   '2026-09-09', 'not_applicable_prod'),
  ('hallucination medium', 'pctile < 0.05 AND similarity < 0.75',
   'Dropped: 4.9% false positive rate on computed numbers.', '2026-08-20', 'not_applicable_prod'),
  ('hallucination_similarity_cross_check', 'resp_vs_prompt_similarity < 0.80',
   'Dropped with hallucination detector.', '2026-09-05', 'not_applicable_prod'),
  ('hallucination_unverified_rate', 'count>=10 AND unverified_rate > 0.5, 24h window',
   'Dropped with hallucination detector.', '2026-09-09', 'not_applicable_prod'),
  ('latency_anomaly', 'p95 + 3*IQR, is_reliable only',
   'Dropped: no latency_ms in prod. Phase 2 via dev enrichment.', '2026-08-20', 'not_applicable_prod'),
  ('latency_anomaly_findings_min_count', '>= 2 anomalous calls in the detection window',
   'Dropped with latency_anomaly.', '2026-09-04', 'not_applicable_prod'),
  ('latency_anomaly_unreliable_baseline', 'same anomaly verdicts, is_reliable = false',
   'Dropped with latency_anomaly.', '2026-09-09', 'not_applicable_prod'),
  ('latency_baseline_missing', 'successful calls exist with no capability_latency_baseline row',
   'Dropped: no latency_ms in prod.', '2026-09-09', 'not_applicable_prod'),
  ('latency_baseline_n_samples', '>= 30',
   'Dropped with latency_anomaly.', '2026-09-05', 'not_applicable_prod'),
  ('latency_baseline_span_days', '>= 6 distinct calendar days with a call',
   'Dropped with latency_anomaly.', '2026-09-05', 'not_applicable_prod'),
  ('latency_fixed_ceiling', 'latency_ms > 30000, successful calls',
   'Dropped: no latency_ms in prod.', '2026-09-09', 'not_applicable_prod'),
  ('prompt_size_drift_multiplier', '2x nightly baseline p95 (prompt or response chars)',
   'Dropped: no user_prompt/system_prompt in prod.', '2026-09-04', 'not_applicable_prod'),
  ('rapid_human_correction',
   'AI publish and ISH correction match within 10 minutes',
   'Dropped: 6 rows total, dormant, research signal.', '2026-09-01', 'not_applicable_prod'),
  ('runtime_allowlist_populated', '0 rows in runtime_allowlist for the running environment',
   'Dropped: single transport (cortex), zero violations.', '2026-09-09', 'not_applicable_prod'),
  ('runtime_observed_digest_floor', 'occurrences_7d >= 5',
   'Dropped with weekly runtime digest.', '2026-08-27', 'not_applicable_prod'),
  ('runtime_violation_digest_cadence', 'weekly, Monday 08:00 America/Indianapolis',
   'Dropped with weekly runtime digest.', '2026-08-27', 'not_applicable_prod'),
  ('runtime_violation_immediate', 'transport OR model_config sanctioned for no capability in env',
   'Dropped: single transport.', '2026-08-27', 'not_applicable_prod'),
  ('ungrounded_token_count', '> 3 AND is_gxp_relevant',
   'Dropped with hallucination detector.', '2026-08-20', 'not_applicable_prod'),
  ('write_lag', 'p95_ingest_only_s > 10',
   'Dropped: write_lag_daily depends on latency_ms for ingest-only computation.', '2026-08-20', 'not_applicable_prod')
AS t(check_name, threshold, basis, set_on, status)
LEFT ANTI JOIN mq_gmdf_dev.oil_obs.threshold_basis existing
  ON existing.check_name = t.check_name;

In [ ]:
%sql
-- ONE-TIME cleanup, safe to re-run (all DROPs are IF EXISTS). Removes tables and views orphaned
-- by the prod migration: detector output tables from dropped detectors, reference tables that are
-- no longer read, and older v1.1 orphans. Order: views first, then tables.

-- views
DROP VIEW IF EXISTS mq_gmdf_dev.oil_obs.v_ungrounded_tokens;

-- detector output tables from dropped detectors
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.hallucination_signal;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.faithfulness_scores;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.latency_anomalies;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.latency_anomaly_findings;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.capability_error_rate_alert;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.capability_error_rate_findings;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.capability_health;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.credential_fastfail_daily;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.latency_failures;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.prompt_size_drift;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.write_lag_daily;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.transport_violation_signatures;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.ish_entity_dim;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.rapid_human_correction;

-- reference tables no longer read by any active detector
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.runtime_observed;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.runtime_allowlist;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs._obs_watermark;

-- older v1.1 orphans (kept here for completeness)
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.hallucination_verdicts;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.blank_output_incidents;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.transport_violations;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.capability_outage_findings;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.response_schema_baseline;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.transport_allowlist;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.success_rate_daily;